In [0]:
--     customer_id STRING,
--     customer_name STRING,
--     email STRING,
--     city STRING,
--     loyalty_tier STRING,
--     join_date DATE,
--     total_orders BIGINT, 
--     lifetime_spend DECIMAL(12,2),
--     avg_order_value DECIMAL(10,2),
--     last_order_date DATE,
--     favorite_restaurant STRING,
--     favorite_item STRING,
--     avg_rating_given DECIMAL(3,2),
--     total_reviews BIGINT,
--     is_at_risk BOOLEAN,  -- No order in 90+ days

In [0]:
create or refresh materialized view restaurant.gold.d_customer_360_sql
comment "this table contains customer details"
tblproperties  ("quality" = "gold")
as
with cte as(
  select 
       o.customer_id,
       r.name as restaurant_name,
       count(distinct o.order_id) as total_orders
from restaurant.silver.fact_orders o 
        join restaurant.silver.dim_restaurants r on o.restaurant_id = r.restaurant_id
        group by all
),
cte2 as(
  select 
       o.customer_id,
       i.item_name,
       sum(i.quantity) as total_orders
   from restaurant.silver.fact_orders o 
        join restaurant.silver.fact_order_items i on o.order_id = i.order_id
        group by all
)
,restau as(
    select 
  customer_id,
  restaurant_name
  from cte c
  qualify row_number() over (partition by customer_id order by total_orders desc) = 1
)
,item as(
    select 
  customer_id,
  item_name
  from cte2
  qualify row_number() over (partition by customer_id order by total_orders desc) = 1
) 
,final as(
  select r.customer_id,r.restaurant_name,i.item_name 
  from restau r join item i on r.customer_id = i.customer_id
)
select 
c.customer_id,
c.name as customer_name,
c.city,
c.join_date,
count(o.order_id) as total_orders,
cast(sum(o.total_amount) as decimal(12,2)) as lifetime_spend,
cast(avg(o.total_amount) as decimal(10,2)) as avg_order_value,
max(o.order_date) as last_order_date,
case when sum(o.total_amount) > 5000 then "Platinum"
     when sum(o.total_amount) > 2000 then "Gold"
     when sum(o.total_amount) > 1000 then "Silver"
     else "Bronze" end as loyality_tier,
cast(avg(r.rating) as decimal(3,2)) as avg_rating_given,
count(r.review_id) as total_reviews_given,
max(f.restaurant_name) as favorite_restaurant,
max(f.item_name) as favorite_item,
case when datediff(current_date(),max(o.order_date)) > 90 then True else False end as is_at_risk
from restaurant.silver.dim_customers c 
     left join restaurant.silver.fact_orders o on c.customer_id = o.customer_id
     left join restaurant.silver.fact_reviews r on c.customer_id = r.customer_id
     left join final f on c.customer_id = f.customer_id
group by all

